# unbox-args-tensor-to-array — worked example 3: Unbox a nested list of MiniTensors

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `unbox-args-tensor-to-array`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

Ops like `cat`/`stack` receive a list of tensors as a single argument. Unboxing must recurse one level into list/tuple args, replacing inner `MiniTensor`s with their `.array` and preserving the container type, so the raw `cat` sees a plain list of arrays rather than a list of wrappers.

## Worked solution

We unbox arguments that may themselves be lists/tuples of tensors.

1. A helper `_maybe_unbox` handles one value: unbox a `MiniTensor`, recurse one level into a `list` or `tuple` preserving its type, and pass anything else through.
2. At the top level we map `_maybe_unbox` over every positional arg.
3. A list arg like `[m1, m2]` becomes `[m1.array, m2.array]` — still a list — so `cat` receives the signature it expects.
4. Mixed inner contents (a tensor next to an int) are handled element-by-element.

We print an unboxed nested list to show the inner tensors were converted while the container stayed a list.

In [ ]:
class MiniTensor:
    def __init__(self, array):
        self.array = array

def unbox_args_nested(args: tuple) -> tuple:
    def _maybe_unbox(a):
        if isinstance(a, MiniTensor):
            return a.array
        if isinstance(a, list):
            return [_maybe_unbox(x) for x in a]
        if isinstance(a, tuple):
            return tuple(_maybe_unbox(x) for x in a)
        return a
    return tuple(_maybe_unbox(a) for a in args)

m1, m2 = MiniTensor([1, 2]), MiniTensor([3, 4])
out = unbox_args_nested(([m1, m2], 0))
print('unboxed:', out)
print('container is list:', isinstance(out[0], list))
print('inner unboxed:', out[0][0] is m1.array)